# Make subsets

use multilabel stratifier, but we'll assign selected samples to be in the last subfold using the quality indicator, thus ensuring that all subsets will have at least 1 of each label

In [ ]:
import os
import numpy as np
import pandas as pd

from pass_pclr.defines import MIMIC_TARGETS

from _multilabel_stratified_sampling import stratify

dataset_path = "/opt/gpudata/ecg/mimic-iv-ecg"
subset_root = "/opt/gpudata/ecg/temp"

labels = MIMIC_TARGETS

In [ ]:
_df = pd.read_csv(os.path.join(dataset_path, "ed-ecgs.csv"))
train_df = _df[_df["split"] == "train"].reset_index(drop=True)
val_test_df = _df[_df["split"].isin(["val", "test"])].reset_index(drop=True)

In [ ]:
per_ecg_labels = train_df[labels]
assert (train_df["age"] < 300).all()
binned_age = pd.cut(train_df["age"], [0, 20, 40, 60, 80, 300])
per_ecg_age = pd.get_dummies(binned_age).astype(int)
per_ecg_sex = pd.get_dummies(train_df["gender"]).astype(int)
per_ecg_stratifier = pd.concat([per_ecg_sex, per_ecg_age, per_ecg_labels], axis=1)

# get per-patient labels
stratifier = per_ecg_stratifier.groupby(train_df["subject_id"]).max()

num_ecgs_per_pt = train_df.groupby("subject_id").size()
assert (num_ecgs_per_pt.index == stratifier.index).all()

In [ ]:
n_patients, n_classes = stratifier.shape

pts = stratifier.index
pt_to_idx = {pid: idx for idx, pid in enumerate(pts)}

# to ensure the train set always has every possible label, we hijack the stratifier's
# notion of quality to ensure that the final split has those patients/ecgs
rng = np.random.default_rng(seed=42)
selected_idxs = set()
for target in labels:
    # this gets us indices in the subset
    candidates = np.argwhere(train_df[target]).flatten()
    selected = rng.choice(candidates)
    # we need to know the index in the stratifier (which has been sorted by patient ID now)
    idx = pt_to_idx[train_df.iloc[selected]["subject_id"]]
    selected_idxs.add(idx)
assert len(selected_idxs) < 256 # should not be larger than smallest subset we aim to make
qualities = [4 if idx in selected_idxs else 2 for idx in range(n_patients)]

label_lists = [np.where(row)[0].tolist() for row in stratifier.to_numpy()]

# aiming for 256 samples per split given 78470 total train samples
# NOTE that the stratifier operates over n_patients, but we'll need to map back
# to n_ecgs for these splits to have the intended size (may not be exact,
# depends on the number of ECGs for the patients selected in each fold)
n_folds = 307
fold_frac = .00326

stratified_ids, stratified_labels = stratify(
    data=label_lists,
    classes=list(range(n_classes)),
    ratios=[fold_frac] * n_folds, 
    qualities=qualities,
    ecgs_per_patient=num_ecgs_per_pt.to_list(),
    nr_clean_folds=1,
    random_seed=42,
)

assert n_patients == sum(len(x) for x in stratified_ids)

In [ ]:
# map folds back to ECGs
# stratifier idx --> patient id --> ECG idx in train_df
pid_to_train_df_idxs = {
    k: list(v) # these are train_df index values (of the pandas dataframe)
    for k, v in train_df.groupby("subject_id").groups.items()
}
stratified_ecgs = []
for idxs in stratified_ids:
    _ecg_idxs = []
    for idx in idxs:
        pid = pts[idx]
        train_df_idxs = pid_to_train_df_idxs[pid]
        _ecg_idxs.extend(train_df_idxs)
    stratified_ecgs.append(_ecg_idxs)

In [ ]:
# check if the last nested fold has all the labels (ensures all larger folds will also have)
assert (train_df.loc[stratified_ecgs[-1], labels].sum() > 0).all()
print(len(stratified_ecgs[-1]), "ECGs in last nested subset")

In [ ]:
def make_train_subset(subfolds: list[int], name: str) -> pd.DataFrame:
    # combine subfolds
    train_idxs = [x for subfold in subfolds for x in stratified_ecgs[subfold]]
    print(len(train_idxs))
    subset_df = pd.concat([train_df.loc[train_idxs], val_test_df], ignore_index=True)

    subset_path = os.path.join(subset_root, f"mimic-iv-ecg-{name}")
    os.makedirs(subset_path, exist_ok=True)

    subset_df.to_csv(os.path.join(subset_path, "ed-ecgs.csv"), index=False)

    # link source data
    for p in [
        "files",
        "machine_measurements.csv",
    ]:
        os.symlink(
            src=os.path.join(dataset_path, p),
            dst=os.path.join(subset_path, p),
        )

    return subset_df

In [ ]:
fold_idxs = np.arange(len(stratified_ecgs))
for name, folds in [
    ("32k", fold_idxs[-128:]),
    ("16k", fold_idxs[-64:]),
    ("8k", fold_idxs[-32:]),
    ("4k", fold_idxs[-16:]),
    ("2k", fold_idxs[-8:]),
    ("1k", fold_idxs[-4:]),
    ("512", fold_idxs[-2:]),
    ("256", fold_idxs[-1:]),
]:
    make_train_subset(list(folds), name)